In [36]:
import numpy as np
import polars as pl
from lets_plot import *
LetsPlot.setup_html()

In [2]:
t = pl.Series("value", [1.0, 2.0, 2.0, 3.0, 5.0])

In [3]:
ftab = t.value_counts()
ftab

value,count
f64,u32
5.0,1
1.0,1
3.0,1
2.0,2


In [4]:
ggplot(ftab, aes(x="value", y="count")) + \
    geom_bar(stat="identity") + \
    labs(x="Value", y="Frequency")        

In [5]:
ftab.get_column("value")

value
f64
5.0
1.0
3.0
2.0


In [6]:
ftab.get_column("count")

count
u32
1
1
1
2


In [7]:
for x, freq in ftab.rows():
    print(x, freq)

5.0 1
1.0 1
3.0 1
2.0 2


In [8]:
preg = pl.read_parquet("../data/2002FemPregCleaned.parquet")

In [9]:
live = preg.filter(pl.col("outcome") == 1)

In [10]:
ftab_lb = preg.get_column("birthwgt_lb").drop_nulls().value_counts()

In [11]:
ggplot(ftab_lb, aes(x="birthwgt_lb", y="count")) + \
    geom_bar(stat="identity") + \
    labs(x="Pounds", y="Frequency")

In [12]:
ftab_lb.item(ftab_lb.get_column("count").arg_max(), "birthwgt_lb")

7.0

In [13]:
ftab_oz = live.get_column("birthwgt_oz").drop_nulls().value_counts()

ggplot(ftab_oz, aes(x="birthwgt_oz", y="count")) + \
    geom_bar(stat="identity") + \
    labs(x="Ounces", y="Frequency")

In [14]:
ftab_age = live.get_column("agepreg").drop_nulls().value_counts()

In [15]:
ggplot(ftab_age, aes(x="agepreg", y="count")) + \
    geom_bar(stat="identity", size=0.1) + \
    labs(x="Age", y="Frequency")

In [16]:
ftab_length = live.get_column("prglngth").drop_nulls().value_counts().sort(by="prglngth", descending=False)

ggplot(ftab_length, aes(x="prglngth", y="count")) + \
    geom_bar(stat="identity") + \
    xlim(20, 50) + \
    labs(x="Weeks", y="Frequency")

In [17]:
ftab_length.head(10)

prglngth,count
i64,u32
0,1
4,1
9,1
13,1
17,2
18,1
19,1
20,1
21,2


In [18]:
ftab_length.tail(10)

prglngth,count
i64,u32
40,1116
41,587
42,328
43,148
44,46
45,10
46,1
47,1
48,7


In [19]:
firsts = live.filter(pl.col("birthord") == 1)
others = live.filter(pl.col("birthord") != 1)

In [20]:
ftab_firsts = firsts.get_column("prglngth").value_counts().sort(by="prglngth", descending=False)
ftab_others = others.get_column("prglngth").value_counts().sort(by="prglngth", descending=False)

In [21]:
def two_bar_plots(name1, ftab1, name2, ftab2, *, xlabel=None, ylabel=None, xlm=None):
    cats = pl.Categories()
    ftab1 = ftab1.with_columns(category=pl.lit(name1).cast(pl.Categorical(cats)))
    ftab2 = ftab2.with_columns(category=pl.lit(name2).cast(pl.Categorical(cats)))
    ftab = ftab1.vstack(ftab2)

    fig = ggplot(ftab, aes(x=ftab.columns[0], y=ftab.columns[1], fill=ftab.columns[2])) + \
        geom_bar(stat="identity", position="dodge") + \
        labs(fill="")

    if xlabel:
        fig += labs(x=xlabel)

    if ylabel:
        fig += labs(y=ylabel)

    if xlim:
        fig += xlim(xlm[0], xlm[1])

    return fig
    

In [22]:
two_bar_plots("firsts", ftab_firsts, "others", ftab_others, xlabel="Weeks", ylabel="Frequency", xlm=[20, 50])

In [23]:
firsts.height, others.height

(4413, 4735)

In [26]:
first_mean = firsts.get_column("prglngth").mean()
other_mean = others.get_column("prglngth").mean()
first_mean, other_mean

(38.60095173351461, 38.52291446673706)

In [28]:
diff = first_mean - other_mean
diff, diff * 7 * 24

(0.07803726677754952, 13.11026081862832)

In [29]:
diff / live.get_column("prglngth").mean() * 100

0.20237586646738304

In [30]:
diff / live.get_column("prglngth").std()

0.028877623375210403

In [31]:
group1, group2 = firsts.get_column("prglngth"), others.get_column("prglngth")

In [32]:
v1, v2 = group1.var(), group2.var()

In [34]:
n1, n2 = len(group1), len(group2)
pooled_var = (n1 * v1 + n2 * v2) / (n1 + n2)

In [37]:
np.sqrt(pooled_var)

np.float64(2.7022108144953867)

In [38]:
firsts.get_column("prglngth").std(), others.get_column("prglngth").std()

(2.7919014146687204, 2.615852350439238)

In [39]:
def cohen_effect_size(group1, group2):
    diff = group1.mean() - group2.mean()
    v1, v2 = group1.var(), group2.var()
    n1, n2 = len(group1), len(group2)
    pooled_var = (n1 * v1 + n2 * v2) / (n1 + n2)
    return diff / np.sqrt(pooled_var)

In [41]:
cohen_effect_size(firsts.get_column("prglngth"), others.get_column("prglngth"))

np.float64(0.02887904465444983)

## Exercises

For the exercises in this chapter, we'll load the NSFG female respondent file, which contains one row for each female respondent.
Instructions for downloading the data and the codebook are in the notebook for this chapter.

In [42]:
resp = pl.read_parquet("../data/2002FemResp.parquet")
resp.shape

(7643, 3092)

This `DataFrame` contains 3092 columns, but we'll use just a few of them.

### Exercise 2.1

We'll start with `totincr`, which records the total income for the respondent's family, encoded with a value from 1 to 14.
You can read the codebook for the respondent file to see what income level each value represents.

Make a `FreqTab` object to represent the distribution of this variable and plot it as a bar chart.

In [43]:
ftab_income = resp.get_column("totincr").drop_nulls().value_counts()

ggplot(ftab_income, aes(x="totincr", y="count")) + \
    geom_bar(stat="identity") + \
    labs(x="Income (category)", y="Frequency")

### Exercise 2.2

Make a frequency table of the `parity` column, which records the number of children each respondent has borne.
How would you describe the shape of this distribution?

In [45]:
ftab_parity = resp.get_column("parity").drop_nulls().value_counts().sort(by="parity", descending=False)

ggplot(ftab_parity, aes(x="parity", y="count")) + \
    geom_bar(stat="identity") + \
    labs(x="Parity", y="Frequency")

The distribution is skewed to the right.

In [47]:
ftab_parity.tail(10)

parity,count
i64,u32
3,828
4,309
5,95
6,29
7,15
8,8
9,2
10,3
16,1


### Exercise 2.3

Let's investigate whether women with higher or lower income bear more children.
Use the query method to select the respondents with the highest income (level 14).
Plot the frequency table of `parity` for just the high income respondents.

In [49]:
rich = resp.filter(pl.col("totincr") == 14)

ftab_parity = rich.get_column("parity").drop_nulls().value_counts()

ggplot(ftab_parity, aes(x="parity", y="count")) + \
    geom_bar(stat="identity") + \
    labs(x="Parity", y="Count")

Compare the mean `parity` for high income respondents and others.

In [50]:
not_rich = resp.filter(pl.col("totincr") < 14)
rich.get_column("parity").mean(), not_rich.get_column("parity").mean()

(1.0758620689655172, 1.2495758136665125)

Compute Cohen's effect size for this difference.
How does it compare with the difference in pregnancy length for first babies and others?

In [52]:
cohen_effect_size(rich.get_column("parity"), not_rich.get_column("parity"))

np.float64(-0.12511855314660367)

In [53]:
# Solution

# The NSFG includes respondents born in different years and
# interviewed at different ages.
# Income and parity depend on both of these factors.

# To check whether people with higher income have more
# children, we need to compare people from the same generation
# interviewed at the same ages.